Literature Download using Elsapy

In [ ]:
from elsapy.elsclient import ElsClient
from elsapy.elsdoc import FullDoc, AbsDoc
import pandas as pd
import json
import os
apikey = "3af51ecbe651ef826516793ab9bbed05"

lit_data = pd.read_excel("HSLA_literature.xlsx")
pii_list = list(lit_data["pii"])
client = ElsClient(apikey)

## ScienceDirect (full-text) document example using PII
for pii in pii_list[:1]:
    pii_doc = FullDoc(sd_pii = pii)
    if pii_doc.read(client):
        print ("pii_doc.title: ", pii_doc.title)
        text = pii_doc.data.get('originalText')
        print(len(text))
        print(text[30000:33000])
        pii_doc.write()   
    else:
        print ("Read document failed.")





pii_doc.title:  Corrosion analysis of HSLA-80/Inconel-625 dissimilar weldments
86923
39]. Macro- and micro-galvanic corrosion prevail, which must be tackled to ensure structural integrity during service. Nevertheless, the corrosion behaviors of heterogeneous dissimilar weldments have been less well studied compared to those of mechanical properties and welding processes. This study aimed to investigate the electrochemical properties of HSLA-80/Inconel-625 dissimilar weldments, focusing on the galvanic coupling effect, which impacts their corrosion behavior and performance in practical applications. The HAZ of HSLA-80 steel was found to be the most vulnerable to corrosion in 0.6 M NaCl solution. Meanwhile, the HAZ of HSLA-80 steel was directly contacted with the most noble fusion zone. This resulted in severe galvanic corrosion in the HAZ of HSLA-80 steel. The corrosion rate measured on the whole dissimilar weldment was 212.8 mpy (mils per year), which is markedly larger than that of th

In [3]:
import re
import json
import pandas as pd

from elsapy.elsclient import ElsClient
from elsapy.elsdoc import FullDoc

from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.langchain import LangchainEmbedding
from langchain_huggingface import HuggingFaceEmbeddings
from llama_index.llms.ollama import Ollama

apikey = "3af51ecbe651ef826516793ab9bbed05"

lit_data = pd.read_excel("HSLA_literature.xlsx")
pii_list = list(lit_data["pii"])
client = ElsClient(apikey)


def extract_text(raw_text):
    if isinstance(raw_text, str):
        return raw_text

    if isinstance(raw_text, dict):
        texts = []

        def recurse(x):
            if isinstance(x, str):
                texts.append(x)
            elif isinstance(x, dict):
                for v in x.values():
                    recurse(v)
            elif isinstance(x, list):
                for item in x:
                    recurse(item)

        recurse(raw_text)
        return " ".join(texts)

    return ""


def clean_elsevier_text(text):
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text).strip()

    for marker in ["References", "REFERENCE", "Bibliography"]:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]
            break

    return text


documents = []

for pii in pii_list:
    pii_doc = FullDoc(sd_pii=pii)

    if pii_doc.read(client):
        print("title:", pii_doc.title)

        raw_text = pii_doc.data.get("originalText", "")
        text = extract_text(raw_text)
        clean_text = clean_elsevier_text(text)

        if len(clean_text) > 3000:
            documents.append(
                Document(
                    text=clean_text,
                    metadata={
                        "title": getattr(pii_doc, "title", ""),
                        "pii": pii
                    }
                )
            )
    else:
        print("Read failed:", pii)


# embedding model
embed_model = LangchainEmbedding(
    HuggingFaceEmbeddings(
        model_name="BAAI/bge-large-en-v1.5",
        encode_kwargs={"normalize_embeddings": True}
    )
)

Settings.embed_model = embed_model
Settings.text_splitter = SentenceSplitter(chunk_size=700, chunk_overlap=120)

# local LLM with Ollama
Settings.llm = Ollama(model="llama3.2:3b", request_timeout=120.0)

# build index
index = VectorStoreIndex.from_documents(documents)

# top-k retrieval + answer generation
query_engine = index.as_query_engine(similarity_top_k=4)

response = query_engine.query(
    "Summarize how heat input and cooling rate affect yield strength in welded steels."
)

print(response)

title: Corrosion analysis of HSLA-80/Inconel-625 dissimilar weldments
title: Numerical investigation of thermal-microstructure behavior in hybrid CMT and tandem P-GMAW-narrow-gap-welding of HSLA-steel
title: Microstructural evolution and precipitation kinetics in Nb–V microalloyed HSLA steel welded joints
title: Flux enhancement with titanium or vanadium oxides addition for superior submerged arc welding of HSLA steel plates
title: Induction assisted autogenous plasma arc welding of HSLA steel 
title: Role of arc rotational speed and post-weld heat treatment on the microstructure and mechanical characteristics of 15CDV6 HSLA steel weld joints made by spin arc GMAW 
title: Experimental and statistical investigation of laser welding with different joint gap widths for HSLA steel
title: Control of post-weld fracture toughness in friction stir processed X80 HSLA steel 
title: Effect of welding processes on high cycle fatigue behavior for naval grade HSLA joints: A fatigue strength predicti